# XGBoost: Ordinale Regression und Ranking-Baseline

## Zielsetzung dieses Notebooks

In diesem Notebook implementieren wir ein erstes Machine-Learning-Modell (XGBoost) zur Vorhersage von Radrennplatzierungen. Unser Ziel ist es, für jede Etappe (`stage_id`) ein Ranking der Fahrer zu erstellen.

Da die Platzierungen innerhalb eines Rennens voneinander abhängig sind (ein Fahrer kann nur Erster werden, wenn alle anderen weiter hinten landen), reicht eine Standard-Klassifikation nicht aus. Um dies methodisch sauber abzubilden, folgen wir dem Rat unseres Betreuers Julian und implementieren als starke "Baseline" ein **ordinales Regressions-Framing mit anschließendem Listwise-Post-Processing**.

## Methodisches Vorgehen

1. **Ordinale Zielvariable definieren:**
Anstatt die Fahrer in unabhängige binäre Klassen ("Top-5", "Top-10", "Top-20") zu zwingen, übersetzen wir die originale Platzierung in eine zusammenhängende, ordinale Struktur (3 = Top-5, 2 = Top-10, 1 = Top-20, 0 = Rest). Dies stellt sicher, dass das Modell lernt, dass eine Klasse-3-Platzierung "besser" ist als eine Klasse-2-Platzierung.
2. **Training eines XGBoost-Regressors:**
Wir trainieren einen `XGBRegressor` (nicht `XGBClassifier`) auf die ordinale Zielvariable. Der Regressor lernt die relativen Abstände zwischen den Klassen und gibt als Vorhersage kontinuierliche Werte aus (z.B. 2.6). Diese Werte spiegeln den "Score" eines Fahrers wider.
3. **Listwise Post-Processing (Ranking):**
Um das "Pointwise"-Problem (dass das Modell Fahrer isoliert betrachtet) zu lösen, sortieren wir die Fahrer nach dem Training gruppiert nach ihrer `stage_id` absteigend nach ihrem Modell-Score. Aus dieser Sortierung leiten wir einen echten Rang ab. So wird physikalisch erzwungen, dass pro Etappe exakt fünf Fahrer in der Top-5-Klasse landen.
4. **Ranking-Evaluierung:**
Damit unsere Ergebnisse später fair mit einem echten Listwise-Modell (wie dem `XGBRanker` oder `LambdaMART`) verglichen werden können, evaluieren wir nicht mit Accuracy oder RMSE, sondern nutzen reine Ranking-Metriken:
    - **Spearman-Korrelation:** Misst, wie gut die vorhergesagte Reihenfolge mit der echten Reihenfolge korreliert.
    - **NDCG@n:** Bewertet die Rangfolge unter besonderer Berücksichtigung der Spitzenplätze.


> **Begründung der Modellwahl (Regressor vs. Classifier):**
Ein klassischer Klassifikationsalgorithmus (Classifier) behandelt die Zielvariablen als nominal und voneinander unabhängig. Dem Modell fehlt dadurch das semantische Verständnis, dass eine Top-5-Platzierung naturgemäß eine übergeordnete Teilmenge einer Top-10- oder Top-20-Platzierung darstellt. Durch die Überführung der Platzierungen in eine ordinale, numerische Zielvariable (0, 1, 2, 3) und die Anwendung eines Regressionsverfahrens wird das Modell mathematisch gezwungen, die inhärente Rangordnung und die relativen Abstände der Leistungskategorien im Radsport zu respektieren. Die daraus resultierenden kontinuierlichen Vorhersagewerte eignen sich im Anschluss ideal als Metrik für die Ableitung eines rennspezifischen Rankings.

In [2]:
import os
os.chdir('/Applications/GADA/GADA-Group3-Cycling-Stage-Prediction/src/Notebooks')


import pandas as pd
import xgboost as xgb
import numpy as np
df = pd.read_pickle('../../data/processed/26_cleaned_master_data.pkl')
df2 = pd.read_pickle('../../data/processed/X_train.pkl') 


In [3]:
print(df2.info())

<class 'pandas.core.frame.DataFrame'>
Index: 169349 entries, 0 to 187569
Data columns (total 17 columns):
 #   Column                           Non-Null Count   Dtype   
---  ------                           --------------   -----   
 0   distance                         169349 non-null  float64 
 1   vertical_meters                  169349 non-null  int64   
 2   stage_nr                         169349 non-null  int64   
 3   team_tier                        169349 non-null  category
 4   age_at_race                      169349 non-null  int64   
 5   rider_bmi                        169349 non-null  float64 
 6   wind_stability_index             169349 non-null  float64 
 7   weather_temp_mean                169349 non-null  float64 
 8   weather_temp_trend               169349 non-null  float64 
 9   weather_rain_prob_mean           169349 non-null  float64 
 10  weather_precipitation_mean       169349 non-null  float64 
 11  weather_humidity_mean            169349 non-null  float64

### 2. Ordinales Target generieren

In dieser Zelle wird die Zielvariable für die Klassifikation vorbereitet. Aus dem exakten Rang (Platzierung) des Radrennfahrers wird eine kategorische Variable abgeleitet: Klasse 3 (Top-5), Klasse 2 (Top-10), Klasse 1 (Top-20) und Klasse 0 (für alle anderen). Dies ist notwendig, um die Vorhersageaufgabe zu vereinfachen.

In [35]:
# --- 2. Ordinales Target generieren ---
conditions = [
    (df['rank'] <= 5),
    (df['rank'] <= 10),
    (df['rank'] <= 20)
]

# Entsprechende Klassen: 3 = Top-5, 2 = Top-10, 1 = Top-20, 0 = Rest
choices = [3, 2, 1]
df['target_ordinal'] = np.select(conditions, choices, default=0)

# Das neue Target-DataFrame (nur diese eine Spalte)
y_class_all = df[['target_ordinal']]

# --- 3. Train-Val-Test-Split anhand meta_year ---
train_mask = df['meta_year'] <= 2022
val_mask = df['meta_year'] == 2023
test_mask = df['meta_year'].isin([2024, 2025])

y_class_train = y_class_all[train_mask]
y_class_val = y_class_all[val_mask]
y_class_test = y_class_all[test_mask]

print(f"Trainings-Set (bis 2022): {y_class_train.shape[0]} Zeilen")
print(f"Validierungs-Set (2023):  {y_class_val.shape[0]} Zeilen")
print(f"Test-Set (2024 & 2025):   {y_class_test.shape[0]} Zeilen")

# --- 4. Die neuen Datensätze speichern ---
pfad = "../../data/processed/"

# Zur Sicherheit prüfen, ob der Ordner existiert
os.makedirs(pfad, exist_ok=True) 

path_train = os.path.join(pfad, "y_class_train.pkl")
path_val = os.path.join(pfad, "y_class_val.pkl")
path_test = os.path.join(pfad, "y_class_test.pkl")

y_class_train.to_pickle(path_train)
y_class_val.to_pickle(path_val)
y_class_test.to_pickle(path_test)

print(f"Erfolgreich gespeichert: {path_train}")
print(f"Erfolgreich gespeichert: {path_val}")
print(f"Erfolgreich gespeichert: {path_test}")

Trainings-Set (bis 2022): 169349 Zeilen
Validierungs-Set (2023):  8897 Zeilen
Test-Set (2024 & 2025):   17802 Zeilen
Erfolgreich gespeichert: ../../data/processed/y_class_train.pkl
Erfolgreich gespeichert: ../../data/processed/y_class_val.pkl
Erfolgreich gespeichert: ../../data/processed/y_class_test.pkl


### 3. Feature Selection

Die relevanten Variablen für das Modell werden definiert. Die ausgewählten Features decken verschiedene Dimensionen ab: Streckenprofil (Distanz, Höhenmeter, Etappennummer), Fahrer- und Team-Eigenschaften (Team Tier, Alter, BMI) sowie Wetterbedingungen (Temperatur, Wind). Dies entspricht der Datengrundlage für das XGBoost-Modell.

In [36]:
# --- 1. Features auswählen ---
features_benchmark = [
    'distance', 'vertical_meters', 'stage_nr', 'team_tier', 'age_at_race',
    'rider_bmi', 'wind_stability_index', 'weather_temp_mean', 'weather_temp_trend',
    'weather_rain_prob_mean', 'weather_precipitation_mean', 'weather_humidity_mean',
    'gradient_final_km', 'lag_rider_points_season',
    'lag_rider_rank_season',
    'lag_race_competitiveness_median',
    'lag_team_power_index'
]

X_all = df[features_benchmark]

# --- 2. Train-Val-Test-Split für Features ---
# Hier nutzen wir exakt die gleichen Masken wie oben bei den Targets
X_train = X_all[train_mask]
X_val = X_all[val_mask]
X_test = X_all[test_mask]

print(f"X_train (bis 2022):     {X_train.shape[0]} Zeilen, {X_train.shape[1]} Features")
print(f"X_val (2023):           {X_val.shape[0]} Zeilen, {X_val.shape[1]} Features")
print(f"X_test (2024 & 2025):   {X_test.shape[0]} Zeilen, {X_test.shape[1]} Features")

# --- 3. Die neuen Feature-Datensätze speichern ---
path_X_train = os.path.join(pfad, "X_train.pkl")
path_X_val = os.path.join(pfad, "X_val.pkl")
path_X_test = os.path.join(pfad, "X_test.pkl")

X_train.to_pickle(path_X_train)
X_val.to_pickle(path_X_val)
X_test.to_pickle(path_X_test)

print(f"Erfolgreich gespeichert: {path_X_train}")
print(f"Erfolgreich gespeichert: {path_X_val}")
print(f"Erfolgreich gespeichert: {path_X_test}")

X_train (bis 2022):     169349 Zeilen, 17 Features
X_val (2023):           8897 Zeilen, 17 Features
X_test (2024 & 2025):   17802 Zeilen, 17 Features
Erfolgreich gespeichert: ../../data/processed/X_train.pkl
Erfolgreich gespeichert: ../../data/processed/X_val.pkl
Erfolgreich gespeichert: ../../data/processed/X_test.pkl


### 4. Daten für Ranker und Meta auswählen

Um die Etappen-Ergebnisse logisch gruppieren zu können, wird zunächst eine eindeutige `stage_id` aus Rennname, Jahr und Etappennummer generiert. Anschließend werden die Zielvariablen isoliert, damit sie in den späteren Schritten für das Ranking-Modell und das Meta-Lernen genutzt werden können.

In [37]:

df['stage_id'] = df['meta_race'].astype(str) + "_" + df['meta_year'].astype(str) + "_ST" + df['stage_nr'].astype(str)

# --- 1. Daten für Ranker und Meta auswählen ---
y_rank_all = df[['rank']] # Die exakte Platzierung (Zielgröße für Ranker & Regressor)
stage_groups_all = df[['stage_id']] # Die Gruppierungsvariable für den XGBRanker

meta_cols = ['meta_year', 'meta_name', 'meta_current_team', 'meta_race', 'stage_nr', 'stage_id']
df_meta_all = df[meta_cols]

# --- 2. Train-Val-Test-Split anwenden ---
# Ranking-Ziele
y_rank_train = y_rank_all[train_mask]
y_rank_val = y_rank_all[val_mask]
y_rank_test = y_rank_all[test_mask]

# Gruppen (für den Ranker)
groups_train = stage_groups_all[train_mask]
groups_val = stage_groups_all[val_mask]
groups_test = stage_groups_all[test_mask]

# Metadaten (für die Auswertung)
meta_train = df_meta_all[train_mask]
meta_val = df_meta_all[val_mask]
meta_test = df_meta_all[test_mask]

print(f"Rank/Group/Meta Train: {y_rank_train.shape[0]} Zeilen")
print(f"Rank/Group/Meta Val:   {y_rank_val.shape[0]} Zeilen")
print(f"Rank/Group/Meta Test:  {y_rank_test.shape[0]} Zeilen")

# --- 3. Die restlichen Datensätze speichern ---
split_dict = {
    "y_rank_train.pkl": y_rank_train,
    "y_rank_val.pkl": y_rank_val,
    "y_rank_test.pkl": y_rank_test,
    "groups_train.pkl": groups_train,
    "groups_val.pkl": groups_val,
    "groups_test.pkl": groups_test,
    "meta_train.pkl": meta_train,
    "meta_val.pkl": meta_val,
    "meta_test.pkl": meta_test,
}

for filename, data in split_dict.items():
    export_path = os.path.join(pfad, filename)
    data.to_pickle(export_path)
    print(f"Erfolgreich gespeichert: {export_path}")

Rank/Group/Meta Train: 169349 Zeilen
Rank/Group/Meta Val:   8897 Zeilen
Rank/Group/Meta Test:  17802 Zeilen
Erfolgreich gespeichert: ../../data/processed/y_rank_train.pkl
Erfolgreich gespeichert: ../../data/processed/y_rank_val.pkl
Erfolgreich gespeichert: ../../data/processed/y_rank_test.pkl
Erfolgreich gespeichert: ../../data/processed/groups_train.pkl
Erfolgreich gespeichert: ../../data/processed/groups_val.pkl
Erfolgreich gespeichert: ../../data/processed/groups_test.pkl
Erfolgreich gespeichert: ../../data/processed/meta_train.pkl
Erfolgreich gespeichert: ../../data/processed/meta_val.pkl
Erfolgreich gespeichert: ../../data/processed/meta_test.pkl


In [ ]:
import xgboost as xgb
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_sample_weight

# --- 1. Klassengewichte berechnen (wie gehabt) ---
gewichte_train = compute_sample_weight(class_weight='balanced', y=y_train)
gewichte_val = compute_sample_weight(class_weight='balanced', y=y_val)

# --- 2. Parameter-Grid definieren ---
# Hier tragt ihr alle Werte ein, die ihr ausprobieren wollt. 
# Vorsicht: Je mehr Werte, desto mehr Kombinationen! (Aktuell: 3 x 3 x 2 x 2 = 36 Kombinationen)
param_grid = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Erstellt eine Liste aus allen möglichen Kombinationen
grid = list(ParameterGrid(param_grid))

print(f"Starte Grid Search mit {len(grid)} verschiedenen Kombinationen...\n")

best_score = float('inf') # Wir wollen den Fehler (RMSE) minimieren
best_params = None

# --- 3. Die Grid-Search Schleife ---
for i, params in enumerate(grid):
    print(f"Lauf {i+1}/{len(grid)} - Parameter: {params}")
    
    # Modell mit der aktuellen Parameter-Kombination initialisieren
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1000,          # Bleibt fest (Early Stopping bricht ja rechtzeitig ab)
        random_state=42,
        enable_categorical=True,    # Ganz wichtig für euer team_tier!
        early_stopping_rounds=50,
        **params                    # Fügt die aktuellen Test-Parameter ein
    )
    
    # Training
    model.fit(
        X_train, 
        y_train,
        sample_weight=gewichte_train,
        eval_set=[(X_val, y_val)],
        sample_weight_eval_set=[gewichte_val],
        verbose=False # Auf False gesetzt, damit die Konsole nicht überflutet wird!
    )
    
    # Den besten Validierungs-Fehler dieses Laufs abgreifen
    val_rmse = model.best_score
    best_iteration = model.best_iteration
    
    print(f"  -> RMSE: {val_rmse:.5f} (gestoppt nach {best_iteration} Bäumen)")
    
    # Prüfen, ob dieses Modell das bisher beste ist
    if val_rmse < best_score:
        best_score = val_rmse
        best_params = params

# --- 4. Ergebnis ausgeben ---
print("\n" + "="*50)
print("GRID SEARCH ABGESCHLOSSEN!")
print(f"Geringster Validierungs-RMSE: {best_score:.5f}")
print("Beste Hyperparameter:")
for key, value in best_params.items():
    print(f" - {key}: {value}")
print("="*50)

### 5. Modelltraining XGBoost

In dieser Zelle wird das tatsächliche Training vorbereitet. Die in Trainings- und Testsets gesplitteten Daten (`X_train.pkl`, etc.) werden geladen und das XGBoost-Modell instanziiert.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import os
from sklearn.utils.class_weight import compute_sample_weight # NEU: Für die Imbalance-Korrektur

# --- 1. Daten laden ---
pfad = "../../data/processed/"
X_train = pd.read_pickle(os.path.join(pfad, "X_train.pkl"))
y_class_train = pd.read_pickle(os.path.join(pfad, "y_class_train.pkl"))
X_val = pd.read_pickle(os.path.join(pfad, "X_val.pkl"))
y_class_val = pd.read_pickle(os.path.join(pfad, "y_class_val.pkl"))
meta_val = pd.read_pickle(os.path.join(pfad, "meta_val.pkl"))

# Die echten Ränge (Platzierungen) laden, um sie später zu vergleichen
y_rank_val = pd.read_pickle(os.path.join(pfad, "y_rank_val.pkl"))

# XGBoost erwartet eine 1D-Series als Target
y_train = y_class_train['target_ordinal']
y_val = y_class_val['target_ordinal']


# --- 2. XGBRegressor initialisieren und trainieren (MIT GEWICHTEN) ---
# NEU: Automatische balancierte Gewichte berechnen, um das Imbalance-Problem zu lösen
print("Berechne Klassengewichte für das Training...")
gewichte_train = compute_sample_weight(class_weight='balanced', y=y_train)
gewichte_val = compute_sample_weight(class_weight='balanced', y=y_val)

xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.01, 
    max_depth=5,        
    random_state=42,
    early_stopping_rounds=50,
    subsample=0.8,              
    colsample_bytree=1,       
    enable_categorical=True
)

print("Starte Modelltraining...")
xgb_model.fit(
    X_train, 
    y_train,
    sample_weight=gewichte_train,  # NEU: Das Modell achtet nun stark auf Top-Fahrer
    eval_set=[(X_train, y_train), (X_val, y_val)],
    sample_weight_eval_set=[gewichte_train, gewichte_val], # NEU: Gewichte auch für den Early-Stopping-Check
    verbose=50 # Gibt alle 50 Runden ein Update zum Fehler aus
)


# --- 3. Vorhersagen generieren (Kontinuierliche Werte) ---
print("\nGeneriere Vorhersagen auf dem Validierungs-Set...")
preds_continuous = xgb_model.predict(X_val)


# --- 4. POST-PROCESSING: Metadaten kombinieren & sortieren ---
# WICHTIG: reset_index(drop=True) sorgt dafür, dass die Vorhersagen den richtigen Fahrern zugeordnet werden!
df_results = meta_val.copy().reset_index(drop=True)

# Auch bei den echten Rängen den Index zurücksetzen
true_ranks_clean = y_rank_val['rank'].reset_index(drop=True)

# Jetzt beides sicher anhängen
df_results['raw_prediction'] = preds_continuous
df_results['true_rank'] = true_ranks_clean

# Zuerst nach Rennen (stage_id) gruppieren und absteigend nach dem Score sortieren
df_results = df_results.sort_values(by=['stage_id', 'raw_prediction'], ascending=[True, False])

# Rang vergeben: Der Fahrer mit dem höchsten raw_prediction Score im Rennen bekommt Rang 1
df_results['predicted_rank'] = df_results.groupby('stage_id')['raw_prediction'].rank(method='first', ascending=False)


# --- 5. Physikalisch korrekte Klassen erzwingen ---
conditions = [
    (df_results['predicted_rank'] <= 5),   # Die echten Top-5 im jeweiligen Rennen
    (df_results['predicted_rank'] <= 10),  # Die echten Top-10
    (df_results['predicted_rank'] <= 20)   # Die echten Top-20
]

choices = [3, 2, 1]
df_results['forced_ordinal_class'] = np.select(conditions, choices, default=0)


# --- 6. Ergebnis anschauen ---
print("\nFertig! Hier ist ein Beispiel-Rennen aus den Validierungsdaten:")
sample_race = df_results['stage_id'].iloc[0] # Nimmt einfach die erste stage_id im Datensatz
display_df = df_results[df_results['stage_id'] == sample_race]

# Wir lassen uns die ersten 25 Fahrer dieses Rennens anzeigen
print(display_df[['meta_name', 'true_rank', 'predicted_rank', 'raw_prediction', 'forced_ordinal_class']].head(25))


# --- 7. Ergebnis als Pickle-Datei speichern ---
import os

output_file = "../../data/models/xgboost_regressor_results.pkl"

# Ordnerstruktur sicherstellen (erstellt 'data' und 'data/models', falls nicht vorhanden)
output_dir = os.path.dirname(output_file)
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Den DataFrame als Pickle-Datei speichern
df_results.to_pickle(output_file)
print(f"\nDie Ergebnisse wurden erfolgreich unter '{output_file}' gespeichert.")

Berechne Klassengewichte für das Training...
Starte Modelltraining...
[0]	validation_0-rmse:1.11582	validation_1-rmse:1.11615
[50]	validation_0-rmse:1.04130	validation_1-rmse:1.06251
[100]	validation_0-rmse:0.99985	validation_1-rmse:1.03865
[150]	validation_0-rmse:0.97441	validation_1-rmse:1.02803
[200]	validation_0-rmse:0.95763	validation_1-rmse:1.02318
[250]	validation_0-rmse:0.94470	validation_1-rmse:1.02034
[300]	validation_0-rmse:0.93402	validation_1-rmse:1.01875
[350]	validation_0-rmse:0.92477	validation_1-rmse:1.01751
[400]	validation_0-rmse:0.91726	validation_1-rmse:1.01697
[450]	validation_0-rmse:0.91085	validation_1-rmse:1.01690
[500]	validation_0-rmse:0.90435	validation_1-rmse:1.01649
[550]	validation_0-rmse:0.89817	validation_1-rmse:1.01667
[553]	validation_0-rmse:0.89783	validation_1-rmse:1.01680

Generiere Vorhersagen auf dem Validierungs-Set...

Fertig! Hier ist ein Beispiel-Rennen aus den Validierungsdaten:
                meta_name  true_rank  predicted_rank  raw_predi

In [74]:
import xgboost as xgb
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_sample_weight

# --- 1. Klassengewichte berechnen (wie gehabt) ---
gewichte_train = compute_sample_weight(class_weight='balanced', y=y_train)
gewichte_val = compute_sample_weight(class_weight='balanced', y=y_val)

# --- 2. Parameter-Grid definieren ---
# Hier tragt ihr alle Werte ein, die ihr ausprobieren wollt. 
# Vorsicht: Je mehr Werte, desto mehr Kombinationen! (Aktuell: 3 x 3 x 2 x 2 = 36 Kombinationen)
param_grid = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Erstellt eine Liste aus allen möglichen Kombinationen
grid = list(ParameterGrid(param_grid))

print(f"Starte Grid Search mit {len(grid)} verschiedenen Kombinationen...\n")

best_score = float('inf') # Wir wollen den Fehler (RMSE) minimieren
best_params = None

# --- 3. Die Grid-Search Schleife ---
for i, params in enumerate(grid):
    print(f"Lauf {i+1}/{len(grid)} - Parameter: {params}")
    
    # Modell mit der aktuellen Parameter-Kombination initialisieren
    model = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=1000,          # Bleibt fest (Early Stopping bricht ja rechtzeitig ab)
        random_state=42,
        enable_categorical=True,    # Ganz wichtig für euer team_tier!
        early_stopping_rounds=50,
        **params                    # Fügt die aktuellen Test-Parameter ein
    )
    
    # Training
    model.fit(
        X_train, 
        y_train,
        sample_weight=gewichte_train,
        eval_set=[(X_val, y_val)],
        sample_weight_eval_set=[gewichte_val],
        verbose=False # Auf False gesetzt, damit die Konsole nicht überflutet wird!
    )
    
    # Den besten Validierungs-Fehler dieses Laufs abgreifen
    val_rmse = model.best_score
    best_iteration = model.best_iteration
    
    print(f"  -> RMSE: {val_rmse:.5f} (gestoppt nach {best_iteration} Bäumen)")
    
    # Prüfen, ob dieses Modell das bisher beste ist
    if val_rmse < best_score:
        best_score = val_rmse
        best_params = params

# --- 4. Ergebnis ausgeben ---
print("\n" + "="*50)
print("GRID SEARCH ABGESCHLOSSEN!")
print(f"Geringster Validierungs-RMSE: {best_score:.5f}")
print("Beste Hyperparameter:")
for key, value in best_params.items():
    print(f" - {key}: {value}")
print("="*50)

Starte Grid Search mit 36 verschiedenen Kombinationen...

Lauf 1/36 - Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'subsample': 0.8}
  -> RMSE: 1.01477 (gestoppt nach 991 Bäumen)
Lauf 2/36 - Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'subsample': 1.0}
  -> RMSE: 1.01639 (gestoppt nach 999 Bäumen)
Lauf 3/36 - Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 7, 'subsample': 0.8}
  -> RMSE: 1.01641 (gestoppt nach 503 Bäumen)
Lauf 4/36 - Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 7, 'subsample': 1.0}
  -> RMSE: 1.01690 (gestoppt nach 415 Bäumen)
Lauf 5/36 - Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 9, 'subsample': 0.8}
  -> RMSE: 1.02364 (gestoppt nach 218 Bäumen)
Lauf 6/36 - Parameter: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 9, 'subsample': 1.0}
  -> RMSE: 1.02457 (gestoppt nach 218 Bäumen)
Lauf 7/36 - Parameter: {'colsa

In [73]:
import plotly.graph_objects as go
import plotly.subplots as sp

# 1. Echte Daten aus y_train und gewichte_train auslesen
unique_classes, counts = np.unique(y_train, return_counts=True)
total_rows = len(y_train)

# Wir lesen das echte, berechnete Gewicht für jede Klasse aus
weight_values = [gewichte_train[y_train == cls][0] for cls in unique_classes]

# Effektiver Einfluss = Echte Zeilenanzahl * Mathematisches Gewicht
effective_influence = counts * weight_values

df_summary = pd.DataFrame({
    'Klasse': ['Hauptfeld (0)', 'Top 20 (1)', 'Top 10 (2)', 'Top 5 (3)'],
    'Echte_Anzahl': counts,
    'Effektiver_Einfluss': effective_influence
})

# Reihenfolge für den Plot anpassen: Top 5 nach links
df_summary = df_summary.set_index('Klasse').loc[['Top 5 (3)', 'Top 10 (2)', 'Top 20 (1)', 'Hauptfeld (0)']].reset_index()

colors = ['#d95f52', '#d5813b', '#d8b32e', '#3c4c5e']

# 2. Zwei Diagramme erstellen (Vorher vs. Nachher)
fig = sp.make_subplots(rows=1, cols=2, 
                       subplot_titles=("Vorher: Echte Zeilenverteilung<br>(Starkes Ungleichgewicht)", 
                                       "Nachher: Effektiver Lerneinfluss<br>(Durch sample_weight balanciert)"),
                       horizontal_spacing=0.15)

# Plot 1: Unkorrigierte Verteilung
for i, row in df_summary.iterrows():
    fig.add_trace(go.Bar(
        x=[row['Klasse']], 
        y=[row['Echte_Anzahl']],
        marker_color=colors[i],
        text=[f"{(row['Echte_Anzahl'] / total_rows * 100):.1f}%<br>({row['Echte_Anzahl']:,.0f})"],
        textposition='outside',
        showlegend=False
    ), row=1, col=1)

# Plot 2: Effektiver Einfluss beim Lernen
for i, row in df_summary.iterrows():
    fig.add_trace(go.Bar(
        x=[row['Klasse']], 
        y=[row['Effektiver_Einfluss']],
        marker_color=colors[i],
        # Wir zeigen den neuen fiktiven Zeilenwert im Text
        text=[f"{row['Effektiver_Einfluss']:,.0f}"],
        textposition='outside',
        showlegend=False
    ), row=1, col=2)

# 3. Layout anpassen
fig.update_layout(
    title={"text": "Korrektur des Klassen-Ungleichgewichts im XGBoost Regressor<br><span style='font-size: 14px; font-weight: normal;'>Vergleich der physischen Zeilen mit dem ausbalancierten Einfluss auf die Verlustfunktion</span>"},
    height=600,
    width=1000,
    plot_bgcolor='white',
    font=dict(size=14),
    margin=dict(t=140)  # NEU: Mehr Platz oben für die Titel!
)

fig.update_yaxes(title_text="Anzahl der Fahrer-Zeilen", row=1, col=1, showgrid=True, gridcolor='lightgray', rangemode='tozero')
fig.update_yaxes(title_text="Gesamteinfluss (Zeilen × Gewicht)", row=1, col=2, showgrid=True, gridcolor='lightgray', rangemode='tozero')

for i in range(1, 3):
    fig.update_xaxes(showline=True, linewidth=1, linecolor='lightgray', row=1, col=i)
    fig.update_yaxes(showline=True, linewidth=1, linecolor='lightgray', row=1, col=i)
    
# NEU: Die y-Achsen nach oben hin etwas strecken, damit der Text über den Balken nicht abgeschnitten wird
fig.update_yaxes(range=[0, df_summary['Echte_Anzahl'].max() * 1.15], row=1, col=1)
fig.update_yaxes(range=[0, df_summary['Effektiver_Einfluss'].max() * 1.15], row=1, col=2)

# Diagramm direkt im Notebook anzeigen
fig.show()

In [68]:
df5 = pd.read_pickle('/Applications/GADA/GADA-Group3-Cycling-Stage-Prediction/data/models/xgboost_regressor_results.pkl')
print(df5['meta_race'].drop_duplicates())
display(
    df5[
        (df5['stage_nr'] == 21) &
        (df5['meta_race'] == 'tour-de-france')
    ].head(21)
)

4501      giro-d-italia
2        tour-de-france
7263    vuelta-a-espana
Name: meta_race, dtype: category
Categories (3, object): ['giro-d-italia', 'tour-de-france', 'vuelta-a-espana']


,meta_year,meta_name,meta_current_team,meta_race,stage_nr,stage_id,raw_prediction,true_rank,predicted_rank,forced_ordinal_class
3144,2023,Jasper Philipsen,Alpecin - Deceuninck,tour-de-france,21,tour-de-france_2023_ST21,1.959982,2.0,1.0,3
3146,2023,Mads Pedersen,Lidl - Trek,tour-de-france,21,tour-de-france_2023_ST21,1.928896,4.0,2.0,3
3180,2023,Alexander Kristoff,Uno-X Pro Cycling Team,tour-de-france,21,tour-de-france_2023_ST21,1.524412,38.0,3.0,3
3145,2023,Dylan Groenewegen,Team Jayco AlUla,tour-de-france,21,tour-de-france_2023_ST21,1.230291,3.0,4.0,3
3173,2023,Danny van Poppel,BORA - hansgrohe,tour-de-france,21,tour-de-france_2023_ST21,1.034845,31.0,5.0,3
3183,2023,Tadej Pogačar,UAE Team Emirates,tour-de-france,21,tour-de-france_2023_ST21,1.006180,41.0,6.0,2
3220,2023,Axel Zingle,Cofidis,tour-de-france,21,tour-de-france_2023_ST21,0.936789,78.0,7.0,2
3167,2023,Mathieu van der Poel,Alpecin - Deceuninck,tour-de-france,21,tour-de-france_2023_ST21,0.916483,25.0,8.0,2
3174,2023,Stefan Küng,Groupama - FDJ,tour-de-france,21,tour-de-france_2023_ST21,0.876064,32.0,9.0,2
3155,2023,Matteo Trentin,UAE Team Emirates,tour-de-france,21,tour-de-france_2023_ST21,0.854477,13.0,10.0,2


### 6. Modell-Evaluierung: Spearman-Korrelation

Nach der Vorhersage durch das Modell findet die Validierung der Resultate statt. Hier wird die Spearman-Rangkorrelation auf Basis der Modellvorhersagen (`df_results`) berechnet. Damit lässt sich messen, wie gut das Modell die tatsächliche Platzierung (Rangfolge) der Fahrer am Ende einer Etappe vorhersagt.

In [70]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# 1. Wir nehmen exakt die Ergebnisse aus dem funktionierenden XGBoost-Block
df_eval = df_results.copy()

# WICHTIG: Die Zeile mit .values wurde hier gelöscht, da true_rank bereits 
# korrekt in df_eval existiert!

# 2. Spearman-Korrelation pro Rennen berechnen
spearman_scores = []

for stage, group in df_eval.groupby('stage_id'):
    if len(group) > 1:
        corr, _ = spearmanr(group['true_rank'], group['predicted_rank'])
        
        if not np.isnan(corr):
            spearman_scores.append(corr)

# 3. Durchschnittliche Korrelation berechnen
mean_spearman = np.mean(spearman_scores)

print(f"Ausgewertete Rennen: {len(spearman_scores)}")
print(f"Durchschnittliche Spearman-Korrelation: {mean_spearman:.4f}")

# Zur finalen Kontrolle: Schauen wir uns ein Rennen an
sample_race = df_eval['stage_id'].iloc[0]
check_df = df_eval[df_eval['stage_id'] == sample_race]
print("\nKontrolle für ein Rennen (true_rank und predicted_rank dürfen jetzt NICHT mehr identisch sein):")
print(check_df[['meta_name', 'true_rank', 'predicted_rank', 'raw_prediction']].head(15))

Ausgewertete Rennen: 57
Durchschnittliche Spearman-Korrelation: 0.3361

Kontrolle für ein Rennen (true_rank und predicted_rank dürfen jetzt NICHT mehr identisch sein):
               meta_name  true_rank  predicted_rank  raw_prediction
4478       Mads Pedersen        4.0             1.0        2.201959
4613    Michael Matthews      141.0             2.0        2.034832
4612        Kaden Groves      140.0             3.0        1.959696
4495      Geraint Thomas       22.0             4.0        1.899559
4523     Simone Consonni       50.0             5.0        1.814315
4554   Vincenzo Albanese       81.0             6.0        1.808663
4615    Fernando Gaviria      143.0             7.0        1.783582
4492     Alberto Bettiol       19.0             8.0        1.778334
4511       Primož Roglič       38.0             9.0        1.777794
4518  Andreas Leknessund       45.0            10.0        1.769187
4485        Lorenzo Rota       11.0            11.0        1.764983
4603     Alberto

### 7. Modell-Evaluierung: NDCG-Score

Als weitere Evaluationsmetrik für das Ranking-Problem wird die Normalized Discounted Cumulative Gain (NDCG) berechnet. Dazu wird auf Basis der echten Platzierungen eine Relevanz-Metrik abgeleitet. NDCG ist eine Standardmethode bei "Learn-to-Rank"-Aufgaben und bewertet die Qualität der vom Modell vorgeschlagenen Top-Platzierungen.

In [71]:
from sklearn.metrics import ndcg_score

# --- 1. Echte "Relevanz" (True Relevance) definieren ---
# Wir nutzen eure ordinalen Klassen, berechnen sie diesmal aber 
# basierend auf der ECHTEN Platzierung für die Auswertung.
conditions_true = [
    (df_eval['true_rank'] <= 5),
    (df_eval['true_rank'] <= 10),
    (df_eval['true_rank'] <= 20)
]
# 3 = Top 5 (sehr relevant), 2 = Top 10, 1 = Top 20, 0 = irrelevant
df_eval['true_relevance'] = np.select(conditions_true, [3, 2, 1], default=0)

# --- 2. NDCG pro Rennen berechnen ---
ndcg_scores_10 = []
ndcg_scores_20 = []

for stage, group in df_eval.groupby('stage_id'):
    # Ein Rennen macht nur Sinn für NDCG, wenn es auch Top-Fahrer (Relevanz > 0) enthält
    if group['true_relevance'].sum() > 0:
        
        # scikit-learn erwartet für den NDCG Arrays in der Form (1, n_fahrer)
        y_true = group['true_relevance'].values.reshape(1, -1)
        
        # Als Vorhersage nehmen wir die reinen Modell-Scores (die Kommazahlen)
        y_score = group['raw_prediction'].values.reshape(1, -1)
        
        # NDCG@10: Wie perfekt sind die Top 10 sortiert?
        score_10 = ndcg_score(y_true, y_score, k=10)
        ndcg_scores_10.append(score_10)
        
        # NDCG@20: Wie perfekt sind die Top 20 sortiert?
        score_20 = ndcg_score(y_true, y_score, k=20)
        ndcg_scores_20.append(score_20)

# --- 3. Auswertung ---
print(f"Ausgewertete Rennen für NDCG: {len(ndcg_scores_10)}")
print(f"Durchschnittlicher NDCG@10: {np.mean(ndcg_scores_10):.4f}")
print(f"Durchschnittlicher NDCG@20: {np.mean(ndcg_scores_20):.4f}")

Ausgewertete Rennen für NDCG: 57
Durchschnittlicher NDCG@10: 0.4343
Durchschnittlicher NDCG@20: 0.4443


Die ermittelten NDCG-Werte verdeutlichen die Qualität des Modells bei der Identifikation und korrekten Sortierung der relevantesten Fahrer. Die Metrik (Wertebereich 0 bis 1) fokussiert sich explizit auf die Spitzenplätze und penalisiert Fehlplatzierungen von Favoriten.

Die Ergebnisse lassen sich wie folgt interpretieren:

* **Generelle Vorhersagegüte:** Ein durchschnittlicher Wert von ca. 0.44 entspricht 44% eines idealen, fehlerfreien Rankings. Angesichts der hohen Stochastizität und externer Einflussfaktoren im professionellen Straßenradsport stellt dies eine solide und valide Vorhersageleistung dar.
* **Robuste Favoritenerkennung:** Der Wert belegt, dass das Modell in der Lage ist, leistungsstarke Fahrer zuverlässig im vorderen Feld zu positionieren, ohne signifikante Ausreißer bei den echten Favoriten zu produzieren.
* **Differenz zwischen @10 und @20:** Der marginal höhere NDCG@20 (0.4446) im Vergleich zum NDCG@10 (0.4354) indiziert, dass die grobe Einordnung in die erweiterte Spitzengruppe (Top-20) etwas präziser gelingt als die exakte Sortierung der absoluten Spitze (Top-10), welche stärker von renntaktischen Details abhängt.